In [ ]:
!pip install deepface

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/dataset_extractedfaces.zip")
extract_to = Path("/content/face_module")

extract_to.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_to)

print("Extracted to:", extract_to)

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("/content/input2400.csv")
dataset_root = Path("/content/dataset_extractedfaces")

df = pd.read_csv(csv_path)
print(df.head())
print("Rows:", len(df))

In [ ]:
# =========================================================
# FaceSM analysis with actual SI_NI_FGSM / SI_NI_FGSM_SM logic
# 1. Embedding trajectory
# 2. Gradient alignment
# Colab-ready, TensorFlow, based on repo logic
# =========================================================

import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from deepface import DeepFace

# -----------------------------
# CONFIG
# -----------------------------
ATTACKER_MODELS = {
    "Facenet512": (160, 160),
    "GhostFaceNet": (112, 112),
    "ArcFace": (112, 112),
    "VGG-Face": (224, 224),
}

CSV_PATH = Path("/content/input2400.csv")
DATASET_ROOT = Path("/content/dataset_extractedfaces")   # change if nested one level deeper
OUT_DIR = Path("/content/sini_facesm_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPSILON = 0.062
NUM_ITER = 12
DECAY = 1.0
SOURCE_LAMBDA = 0.20
SCALES = (1.0, 0.5, 0.25, 0.125, 0.0625)

TRAJ_MODEL = "Facenet512"
TRAJ_ROW_INDEX = 0
ALIGN_ATTACK_TYPE = "impersonation_attack"
ALIGN_N_SAMPLES = 40
SEED = 2027

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams["figure.dpi"] = 200
plt.rcParams["savefig.dpi"] = 300

# -----------------------------
# GPU setup
# -----------------------------
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# -----------------------------
# HELPERS
# -----------------------------
def resolve_dataset_path(raw_path, dataset_root):
    value = str(raw_path).strip()
    if not value:
        return None

    p = Path(value)
    if p.exists():
        return p.resolve()

    parts = list(p.parts)
    if "dataset_extractedfaces" in parts:
        idx = parts.index("dataset_extractedfaces")
        rel_parts = parts[idx + 1:]
        candidate = (dataset_root / Path(*rel_parts)).resolve()
        if candidate.exists():
            return candidate

    candidate = (dataset_root / value).resolve()
    if candidate.exists():
        return candidate

    matches = list(dataset_root.rglob(Path(value).name))
    if len(matches) == 1:
        return matches[0].resolve()

    return None


def load_and_preprocess(path, input_size):
    img = Image.open(path).convert("RGB").resize(input_size)
    arr = np.array(img).astype("float32") / 255.0
    return ((arr - 0.5) * 2.0)[None, ...]


def compute_embedding(model, x, multi_view=False):
    out = model(x, training=False)
    if isinstance(out, (tuple, list)):
        out = out[0]
    emb = tf.nn.l2_normalize(out, axis=1)

    if not multi_view:
        return emb

    x_flip = tf.image.flip_left_right(x)
    out_flip = model(x_flip, training=False)
    if isinstance(out_flip, (tuple, list)):
        out_flip = out_flip[0]
    emb_flip = tf.nn.l2_normalize(out_flip, axis=1)

    return tf.nn.l2_normalize(0.5 * (emb + emb_flip), axis=1)


def attack_loss(cos, attack_type):
    normalized = str(attack_type).strip().lower()
    return tf.reduce_mean(cos if normalized == "impersonation_attack" else (1.0 - cos))


def attack_loss_sm(cos_t, cos_s, attack_type, source_lambda):
    normalized = str(attack_type).strip().lower()
    if normalized == "impersonation_attack":
        score = cos_t - source_lambda * cos_s
    else:
        score = (1.0 - cos_t) + source_lambda * (1.0 - cos_s)
    return tf.reduce_mean(score)


# -----------------------------
# REAL SI_NI_FGSM trajectory
# -----------------------------
def run_si_ni_fgsm_trajectory(model, src, tgt, attack_type):
    adv = tf.identity(src)
    g = tf.zeros_like(src)
    alpha = EPSILON / NUM_ITER

    tgt_emb = tf.nn.l2_normalize(compute_embedding(model, tgt, multi_view=False), axis=1)
    src_emb = tf.nn.l2_normalize(compute_embedding(model, src, multi_view=False), axis=1)

    rows = []

    for t in range(NUM_ITER + 1):
        adv_emb = compute_embedding(model, adv, multi_view=False)
        cos_t = float(tf.reduce_sum(adv_emb * tgt_emb, axis=1).numpy()[0])
        cos_s = float(tf.reduce_sum(adv_emb * src_emb, axis=1).numpy()[0])
        score = cos_t if attack_type == "impersonation_attack" else (1.0 - cos_t)

        rows.append({
            "iter": t,
            "method": "SI_NI_FGSM",
            "cos_target": cos_t,
            "cos_source": cos_s,
            "score": score,
        })

        if t == NUM_ITER:
            break

        nes = adv + DECAY * alpha * g
        grad_sum = tf.zeros_like(src)

        for s in SCALES:
            with tf.GradientTape() as tape:
                tape.watch(nes)
                emb = compute_embedding(model, nes * s, multi_view=False)
                cos = tf.reduce_sum(emb * tgt_emb, axis=1)
                loss = attack_loss(cos, attack_type)
            grad_sum += tape.gradient(loss, nes)

        grad = grad_sum / len(SCALES)
        grad = grad / (tf.reduce_mean(tf.abs(grad)) + 1e-8)
        g = DECAY * g + grad
        adv = adv + alpha * tf.sign(g)
        adv = tf.clip_by_value(adv, src - EPSILON, src + EPSILON)
        adv = tf.clip_by_value(adv, -1.0, 1.0)

    return pd.DataFrame(rows)


def run_si_ni_fgsm_sm_trajectory(model, src, tgt, attack_type):
    adv = tf.identity(src)
    g = tf.zeros_like(src)
    alpha = EPSILON / NUM_ITER

    src_emb_mv = tf.nn.l2_normalize(compute_embedding(model, src, multi_view=True), axis=1)
    tgt_emb_mv = tf.nn.l2_normalize(compute_embedding(model, tgt, multi_view=True), axis=1)

    rows = []

    for t in range(NUM_ITER + 1):
        adv_emb = compute_embedding(model, adv, multi_view=True)
        cos_t = float(tf.reduce_sum(adv_emb * tgt_emb_mv, axis=1).numpy()[0])
        cos_s = float(tf.reduce_sum(adv_emb * src_emb_mv, axis=1).numpy()[0])

        if attack_type == "impersonation_attack":
            score = cos_t - SOURCE_LAMBDA * cos_s
        else:
            score = (1.0 - cos_t) + SOURCE_LAMBDA * (1.0 - cos_s)

        rows.append({
            "iter": t,
            "method": "SI_NI_FGSM_SM",
            "cos_target": cos_t,
            "cos_source": cos_s,
            "score": score,
        })

        if t == NUM_ITER:
            break

        nes = adv + DECAY * alpha * g
        grad_sum = tf.zeros_like(src)

        for s in SCALES:
            with tf.GradientTape() as tape:
                tape.watch(nes)
                emb = compute_embedding(model, nes * s, multi_view=True)
                cos_t_step = tf.reduce_sum(emb * tgt_emb_mv, axis=1)
                cos_s_step = tf.reduce_sum(emb * src_emb_mv, axis=1)
                loss = attack_loss_sm(cos_t_step, cos_s_step, attack_type, SOURCE_LAMBDA)
            grad_sum += tape.gradient(loss, nes)

        grad = grad_sum / len(SCALES)
        grad = grad / (tf.reduce_mean(tf.abs(grad)) + 1e-8)
        g = DECAY * g + grad
        adv = adv + alpha * tf.sign(g)
        adv = tf.clip_by_value(adv, src - EPSILON, src + EPSILON)
        adv = tf.clip_by_value(adv, -1.0, 1.0)

    return pd.DataFrame(rows)


def plot_trajectory(df, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.8), sharex=True)

    styles = {
        "SI_NI_FGSM": {"color": "#4C72B0", "marker": "o"},
        "SI_NI_FGSM_SM": {"color": "#DD8452", "marker": "s"},
    }

    for method in ["SI_NI_FGSM", "SI_NI_FGSM_SM"]:
        sub = df[df["method"] == method]
        axes[0].plot(sub["iter"], sub["cos_target"], linewidth=2.2, marker=styles[method]["marker"],
                     color=styles[method]["color"], label=method)
        axes[1].plot(sub["iter"], sub["cos_source"], linewidth=2.2, marker=styles[method]["marker"],
                     color=styles[method]["color"], label=method)


    axes[0].set_title("Target Similarity")
    axes[1].set_title("Source Similarity")

    for ax in axes:
        ax.set_xlabel("Iteration")
        ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("Cosine Similarity")
    axes[1].set_ylabel("Cosine Similarity")
    axes[0].legend(frameon=True)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


# -----------------------------
# REAL gradient alignment
# -----------------------------
def get_sini_grad(model, src, tgt, attack_type, use_facesm=False, align_size=(112, 112)):
    src = tf.identity(src)

    if use_facesm:
        src_emb_ref = tf.nn.l2_normalize(compute_embedding(model, src, multi_view=True), axis=1)
        tgt_emb_ref = tf.nn.l2_normalize(compute_embedding(model, tgt, multi_view=True), axis=1)
    else:
        tgt_emb_ref = tf.nn.l2_normalize(compute_embedding(model, tgt, multi_view=False), axis=1)

    adv = tf.identity(src)
    g = tf.zeros_like(src)
    alpha = EPSILON / NUM_ITER
    nes = adv + DECAY * alpha * g

    grad_sum = tf.zeros_like(src)

    for s in SCALES:
        with tf.GradientTape() as tape:
            tape.watch(nes)
            if use_facesm:
                emb = compute_embedding(model, nes * s, multi_view=True)
                cos_t = tf.reduce_sum(emb * tgt_emb_ref, axis=1)
                cos_s = tf.reduce_sum(emb * src_emb_ref, axis=1)
                loss = attack_loss_sm(cos_t, cos_s, attack_type, SOURCE_LAMBDA)
            else:
                emb = compute_embedding(model, nes * s, multi_view=False)
                cos = tf.reduce_sum(emb * tgt_emb_ref, axis=1)
                loss = attack_loss(cos, attack_type)
        grad_sum += tape.gradient(loss, nes)

    grad = grad_sum / len(SCALES)
    grad = tf.image.resize(grad, align_size)
    grad = tf.reshape(grad, [-1])
    grad = tf.cast(grad, tf.float32)
    grad = grad / (tf.norm(grad) + 1e-8)
    return grad


def cosine(a, b):
    return float(tf.reduce_sum(a * b).numpy())


def build_model_inputs(pairs_df):
    src_inputs = []
    tgt_inputs = []

    for _, row in pairs_df.iterrows():
        src_path = resolve_dataset_path(row["img1"], DATASET_ROOT)
        tgt_path = resolve_dataset_path(row["img2"], DATASET_ROOT)
        if src_path is None or tgt_path is None:
            continue

        src_pack = {}
        tgt_pack = {}

        for model_name, size in ATTACKER_MODELS.items():
            src_pack[model_name] = tf.convert_to_tensor(load_and_preprocess(src_path, size), dtype=tf.float32)
            tgt_pack[model_name] = tf.convert_to_tensor(load_and_preprocess(tgt_path, size), dtype=tf.float32)

        src_inputs.append(src_pack)
        tgt_inputs.append(tgt_pack)

    return src_inputs, tgt_inputs


def alignment_matrix(models, src_inputs, tgt_inputs, attack_type, use_facesm=False):
    names = list(models.keys())
    rows = []

    for m1 in names:
        for m2 in names:
            vals = []
            for src_pack, tgt_pack in zip(src_inputs, tgt_inputs):
                g1 = get_sini_grad(models[m1], src_pack[m1], tgt_pack[m1], attack_type, use_facesm=use_facesm)
                g2 = get_sini_grad(models[m2], src_pack[m2], tgt_pack[m2], attack_type, use_facesm=use_facesm)
                vals.append(cosine(g1, g2))

            rows.append({
                "model_a": m1,
                "model_b": m2,
                "alignment": float(np.mean(vals)),
                "method": "SI_NI_FGSM_SM" if use_facesm else "SI_NI_FGSM",
            })

    return pd.DataFrame(rows)


def plot_heatmap(df, out_path, title, cmap="YlOrRd", center=None):
    pivot = df.pivot(index="model_a", columns="model_b", values="alignment")
    plt.figure(figsize=(5.8, 4.8))
    ax = sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap=cmap,
        center=center,
        linewidths=0.5,
        cbar_kws={"shrink": 0.9},
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_delta_heatmap(df_v, df_f, out_path):
    pv = df_v.pivot(index="model_a", columns="model_b", values="alignment")
    pf = df_f.pivot(index="model_a", columns="model_b", values="alignment")
    delta = pf - pv

    plt.figure(figsize=(5.8, 4.8))
    ax = sns.heatmap(
        delta,
        annot=True,
        fmt=".3f",
        cmap="coolwarm",
        center=0.0,
        linewidths=0.5,
        cbar_kws={"shrink": 0.9},
    )
    ax.set_title("Alignment Gain (SI_NI_FGSM_SM - SI_NI_FGSM)")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


# -----------------------------
# MAIN
# -----------------------------
def main():
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    # ----- trajectory -----

    df_pairs = pd.read_csv(CSV_PATH)
    df_pairs=df_pairs[df_pairs['dataset']!='digiface']
    row = df_pairs.iloc[TRAJ_ROW_INDEX]

    print("Raw row:")
    print(row[["img1", "img2", "attack_type"]])

    src_path = resolve_dataset_path(row["img1"], DATASET_ROOT)
    tgt_path = resolve_dataset_path(row["img2"], DATASET_ROOT)
    attack_type = str(row["attack_type"]).strip()

    print("Resolved src:", src_path)
    print("Resolved tgt:", tgt_path)

    if src_path is None or tgt_path is None:
        raise FileNotFoundError(f"Could not resolve source/target paths for row {TRAJ_ROW_INDEX}")

    model = DeepFace.build_model(TRAJ_MODEL).model
    input_size = ATTACKER_MODELS[TRAJ_MODEL]

    src = tf.convert_to_tensor(load_and_preprocess(src_path, input_size), dtype=tf.float32)
    tgt = tf.convert_to_tensor(load_and_preprocess(tgt_path, input_size), dtype=tf.float32)

    traj_v = run_si_ni_fgsm_trajectory(model, src, tgt, attack_type)
    traj_f = run_si_ni_fgsm_sm_trajectory(model, src, tgt, attack_type)
    traj_df = pd.concat([traj_v, traj_f], ignore_index=True)

    traj_df.to_csv(OUT_DIR / "sini_embedding_trajectory.csv", index=False)
    plot_trajectory(traj_df, OUT_DIR / "sini_embedding_trajectory.png")

    # ----- alignment -----
    df_align = pd.read_csv(CSV_PATH)
    df_align=df_align[df_align['dataset']!='digiface']
    df_align = df_align[df_align["attack_type"].astype(str).str.strip() == ALIGN_ATTACK_TYPE].copy()

    if len(df_align) > ALIGN_N_SAMPLES:
        df_align = df_align.sample(n=ALIGN_N_SAMPLES, random_state=SEED)

    models = {name: DeepFace.build_model(name).model for name in ATTACKER_MODELS.keys()}
    src_inputs, tgt_inputs = build_model_inputs(df_align)

    df_v = alignment_matrix(models, src_inputs, tgt_inputs, ALIGN_ATTACK_TYPE, use_facesm=False)
    df_f = alignment_matrix(models, src_inputs, tgt_inputs, ALIGN_ATTACK_TYPE, use_facesm=True)

    pd.concat([df_v, df_f], ignore_index=True).to_csv(OUT_DIR / "sini_gradient_alignment.csv", index=False)

    plot_heatmap(df_v, OUT_DIR / "sini_alignment_heatmap.png", "Gradient Alignment: SI_NI_FGSM")
    plot_heatmap(df_f, OUT_DIR / "sini_sm_alignment_heatmap.png", "Gradient Alignment: SI_NI_FGSM_SM")
    plot_delta_heatmap(df_v, df_f, OUT_DIR / "sini_alignment_gain_heatmap.png")

    print("Saved outputs to:", OUT_DIR)


if __name__ == "__main__":
    main()